### Train YOLO Pollinator Detector

Fine-tunes YOLO26n on the annotated dataset using two stages (frozen backbone → full fine-tune).
Images are tiled to 640 × 640 px before training — without tiling, insects are too small to learn from at full 3008 × 1692 px resolution.
> **Optimizer is AdamW (explicit).** Do not change to `'auto'` — Ultralytics `'auto'` silently overrides the learning rate.

**Input** — CVAT YOLO 1.1 export zip (download from CVAT; place at `BASE_DIR/yolo.zip` or set `YOLO_ZIP`)  
**Output** — `outputs/training/model_runs/{RUN_NAME}_{ts}/` · **overwrites** `models/yolo_best.pt`

**Must edit (Cell 2):**

| Variable | Risk if skipped |
|----------|-----------------|
| `YOLO_ZIP` | `FileNotFoundError` at data-prep time |
| `CVAT_CLASSES` | Wrong class-to-index mapping — verify against `obj.data` inside the zip |
| `KEEP_CLASSES` | Must be a subset of `CVAT_CLASSES`; others are filtered out and indices remapped |
| `BASE_DIR` (Cell 1) | Everything fails — change only when running locally |

**Optional (Cell 2):** `MODEL_SIZE` (`'yolo26n.pt'`), `TILE_SIZE` (640), `TILE_OVERLAP` (0.2), `MERGE_TEST_INTO_TRAIN` (False), `SMOKE_TEST` (False — set True for 1-epoch sanity check), `EPOCHS_S1` / `EPOCHS_S2` (40 / 70), `BATCH` (32)


##### Cell 1 — Environment  ← edit `BASE_DIR` for local runs
Sets all paths.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 1 — ENVIRONMENT  ← only edit this cell for paths
# ════════════════════════════════════════════════════════════
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/pollinator-classification')
else:
    BASE_DIR = Path('/Users/lianshi/Downloads/bachelor thesis'
                    '/automated-ecological-image-analysis'
                    '/ml-pipelines/notebooks/pollinator-classification')

MODEL_DIR  = BASE_DIR / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f'Env       : {"Colab" if IN_COLAB else "Local"}')
print(f'BASE_DIR  : {BASE_DIR}  exists={BASE_DIR.exists()}')
print(f'MODEL_DIR : {MODEL_DIR}  exists={MODEL_DIR.exists()}')


##### Cell 2 — Config  ← **edit before training**
Set `YOLO_ZIP`, `CVAT_CLASSES`, `KEEP_CLASSES`, tile settings, and epochs.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 2 — TRAINING CONFIG  ← edit here
# ════════════════════════════════════════════════════════════
import subprocess, shutil, zipfile, json as _json
from pathlib import Path
from datetime import datetime

RUN_NAME     = 'yolo'           # ← optional label appended to timestamp

YOLO_ZIP     = BASE_DIR / 'yolo.zip'  # ← place your CVAT export zip here (or set any path)
EXTRACT_TO   = Path('/content/data') if IN_COLAB else BASE_DIR / 'extracted'
_ts          = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR   = BASE_DIR / 'outputs' / 'training' / 'model_runs' / f'{RUN_NAME}_{_ts}'

# ── Class config ──────────────────────────────────────────
# CVAT_CLASSES must match the order CVAT used when exporting (index 0, 1, …).
# KEEP_CLASSES is the subset you actually want to train on; the rest are
# filtered out and indices are remapped to 0…N-1.
CVAT_CLASSES = ['bumblebee', 'fly', 'butterfly', 'other', 'unsure']
KEEP_CLASSES = ['fly', 'butterfly']

# ── Model ─────────────────────────────────────────────────
MODEL_SIZE   = 'yolo26n.pt'   # Ultralytics 2026 nano — matches colab pipeline

# ── Tile-based training ───────────────────────────────────
# Slices each source image into overlapping tiles so YOLO sees insects at
# their native pixel scale instead of having them downscaled.  Needed when
# source images are much larger than 640 px (e.g. 4K field cameras).
USE_TILES        = True
TILE_SIZE        = 640        # px per tile edge (square)
TILE_OVERLAP     = 0.2        # fraction of tile width/height to overlap
TILE_MIN_AREA    = 0.1        # keep clipped bbox if ≥ 10% of original area survives
# Negatives per labeled tile to keep as background samples.
# {'train': 2, 'val': 5, 'test': True} → tight balance for train,
# 5:1 for val, keep every empty tile for the honest test number.
KEEP_EMPTY_TILES = {'train': 2, 'val': 5, 'test': True}

# ── Data split ────────────────────────────────────────────
# Set True to move test/ images into train/ before patching (maximises
# Colab GPU use).  Keep False to preserve the three-way split.
MERGE_TEST_INTO_TRAIN = False

# ── Hyperparameters ───────────────────────────────────────
IMG_SIZE     = TILE_SIZE if USE_TILES else 640
BATCH        = 32
EPOCHS_S1    = 40   # frozen backbone
EPOCHS_S2    = 70   # full fine-tune
LR_S1        = 1e-3
LR_S2        = 5e-4
SEED         = 42

# ── Smoke test ────────────────────────────────────────────
# Set True for a 1-epoch end-to-end sanity check (fast, no real training).
SMOKE_TEST   = False
if SMOKE_TEST:
    EPOCHS_S1, EPOCHS_S2 = 1, 1

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
_json.dump({
    'run_name': RUN_NAME, 'timestamp': _ts,
    'model_size': MODEL_SIZE,
    'use_tiles': USE_TILES, 'tile_size': TILE_SIZE, 'tile_overlap': TILE_OVERLAP,
    'keep_classes': KEEP_CLASSES,
    'img_size': IMG_SIZE, 'batch': BATCH,
    'epochs_s1': EPOCHS_S1, 'epochs_s2': EPOCHS_S2,
    'lr_s1': LR_S1, 'lr_s2': LR_S2,
}, open(OUTPUT_DIR / 'config.json', 'w'), indent=2)

print(f'Dataset zip : {YOLO_ZIP}  exists={YOLO_ZIP.exists()}')
print(f'Keep classes: {KEEP_CLASSES}')
print(f'Use tiles   : {USE_TILES}  (size={TILE_SIZE}, overlap={TILE_OVERLAP})')
print(f'Run dir     : {OUTPUT_DIR}')


##### Cell 3 — Install dependencies
Installs `ultralytics` on Colab. Skip if already installed locally.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 3 — INSTALL DEPENDENCIES (Colab only)
# ════════════════════════════════════════════════════════════
if IN_COLAB:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'ultralytics'], check=True)
    print('ultralytics installed')


##### Cell 4 — Data preparation
Unpacks CVAT zip, tiles images to 640 × 640 px, filters classes, rewrites `data.yaml`.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 4 — DATA-PREP HELPERS
# ════════════════════════════════════════════════════════════
import random
from pathlib import Path
from PIL import Image

# ── Extract ───────────────────────────────────────────────
def extract_zip(zip_path, extract_to, name):
    target = Path(extract_to) / name
    if target.exists() and any(target.iterdir()):
        print(f'Reusing {target}'); return target
    target.mkdir(parents=True, exist_ok=True)
    local = target.parent / Path(zip_path).name
    shutil.copy(str(zip_path), str(local))
    subprocess.run(['unzip', '-q', '-o', str(local), '-d', str(target)], check=True)
    local.unlink()
    nested = target / name
    if nested.is_dir():
        for item in list(nested.iterdir()): shutil.move(str(item), str(target / item.name))
        nested.rmdir()
    print(f'Extracted: {target}'); return target

# ── Patch labels ──────────────────────────────────────────
def _write_data_yaml(root, names):
    root = Path(root)
    lines = [f'path: {root}']
    for sp in ('train', 'val', 'test'):
        if (root / 'images' / sp).exists(): lines.append(f'{sp}: images/{sp}')
    lines += ['names:'] + [f'  {i}: {n}' for i, n in enumerate(names)]
    (root / 'data.yaml').write_text('\n'.join(lines) + '\n')

def patch_and_write_yaml(root, cvat_classes, keep_classes, config_id):
    root = Path(root); marker = root / '.patched_for'
    if marker.exists() and marker.read_text() == config_id:
        print('Already patched'); _write_data_yaml(root, keep_classes); return
    ci = {n: i for i, n in enumerate(cvat_classes)}
    remap = {ci[n]: ni for ni, n in enumerate(keep_classes) if n in ci}
    n_strip = n_rem = n_img = 0
    for sp in ('train', 'val', 'test'):
        ld = root / 'labels' / sp; id_ = root / 'images' / sp
        if not ld.exists(): continue
        for lf in ld.glob('*.txt'):
            orig = lf.read_text().splitlines(); kept = []
            for line in orig:
                p = line.strip().split()
                if not p: continue
                try: c = int(p[0])
                except ValueError: continue
                if c in remap: p[0] = str(remap[c]); kept.append(' '.join(p))
            n_strip += len(orig) - len(kept)
            if kept: lf.write_text('\n'.join(kept) + '\n')
            else:
                lf.unlink(); n_rem += 1
                if id_.exists():
                    for img in id_.glob(f'{lf.stem}.*'): img.unlink(); n_img += 1
    print(f'Patched: stripped {n_strip} lines, removed {n_rem} labels, {n_img} images')
    _write_data_yaml(root, keep_classes)
    marker.write_text(config_id)

# ── Merge test → train ────────────────────────────────────
def merge_test_into_train(dataset_root):
    root = Path(dataset_root); moved = 0
    for sub in ('images', 'labels'):
        test_dir = root / sub / 'test'; train_dir = root / sub / 'train'
        if not test_dir.exists(): continue
        train_dir.mkdir(parents=True, exist_ok=True)
        for item in list(test_dir.iterdir()):
            shutil.move(str(item), str(train_dir / item.name)); moved += 1
        test_dir.rmdir()
    print(f'Merged {moved} test files into train/' if moved else 'No test files to merge')

# ── Tile helpers ──────────────────────────────────────────
def _tile_origins(extent, tile_size, overlap):
    if extent <= tile_size: return [0]
    stride = max(1, int(tile_size * (1.0 - overlap)))
    starts = list(range(0, extent - tile_size, stride))
    if not starts or starts[-1] != extent - tile_size:
        starts.append(extent - tile_size)
    return starts

def _parse_yolo_line(line, img_w, img_h):
    parts = line.strip().split()
    if not parts: return None
    try:
        cls = int(parts[0])
        cx = float(parts[1]) * img_w; cy = float(parts[2]) * img_h
        w  = float(parts[3]) * img_w; h  = float(parts[4]) * img_h
    except (ValueError, IndexError): return None
    return cls, cx - w/2, cy - h/2, cx + w/2, cy + h/2

def _clip_to_tile(box, tile, min_area):
    cls, x1, y1, x2, y2 = box
    tx1, ty1, tx2, ty2 = tile
    cx1, cy1 = max(x1, tx1), max(y1, ty1)
    cx2, cy2 = min(x2, tx2), min(y2, ty2)
    if cx2 <= cx1 or cy2 <= cy1: return None
    orig = (x2-x1)*(y2-y1); new = (cx2-cx1)*(cy2-cy1)
    if orig > 0 and new/orig < min_area: return None
    return cls, cx1-tx1, cy1-ty1, cx2-tx1, cy2-ty1

def _to_yolo_line(cls, x1, y1, x2, y2, w, h):
    cx=(x1+x2)/2/w; cy=(y1+y2)/2/h; bw=(x2-x1)/w; bh=(y2-y1)/h
    return f'{cls} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}'

def _resolve_empties(keep, n_pos, n_emp):
    if isinstance(keep, bool): return n_emp if keep else 0
    if isinstance(keep, int):  return min(n_emp, n_pos * keep)
    if isinstance(keep, float): return int(round(n_emp * keep))
    raise TypeError(f'Unsupported keep_empty_tiles type: {type(keep)}')

def slice_dataset(dataset_root, output_root, tile_size=640, overlap=0.2,
                  min_area=0.1, keep_empty_tiles=False, seed=42, jpeg_quality=90):
    """Slice a YOLO-format dataset into overlapping tiles."""
    src = Path(dataset_root); dst = Path(output_root); stats = {}
    for split in ('train', 'val', 'test'):
        src_img = src/'images'/split; src_lbl = src/'labels'/split
        if not src_img.exists(): continue
        dst_img = dst/'images'/split; dst_lbl = dst/'labels'/split
        dst_img.mkdir(parents=True, exist_ok=True)
        dst_lbl.mkdir(parents=True, exist_ok=True)
        candidates = {}; n_src = n_pos = n_emp = 0
        for img_path in sorted(src_img.iterdir()):
            if img_path.suffix.lower() not in ('.jpg','.jpeg','.png'): continue
            n_src += 1
            try:
                with Image.open(img_path) as im: im.load(); W, H = im.size
            except Exception as e: print(f'Skip {img_path.name}: {e}'); continue
            lp = src_lbl / f'{img_path.stem}.txt'
            boxes = []
            if lp.exists():
                for line in lp.read_text().splitlines():
                    b = _parse_yolo_line(line, W, H)
                    if b: boxes.append(b)
            per_img = []
            for y0 in _tile_origins(H, tile_size, overlap):
                for x0 in _tile_origins(W, tile_size, overlap):
                    tile = (x0, y0, x0+tile_size, y0+tile_size)
                    clipped = [c for c in (_clip_to_tile(b, tile, min_area) for b in boxes) if c]
                    per_img.append((x0, y0, clipped))
                    if clipped: n_pos += 1
                    else: n_emp += 1
            if per_img: candidates[img_path] = per_img
        keep_for = keep_empty_tiles.get(split, False) if isinstance(keep_empty_tiles, dict) else keep_empty_tiles
        target = _resolve_empties(keep_for, n_pos, n_emp)
        keep_set = set()
        if 0 < target < n_emp:
            all_emp = [(p,x,y) for p,items in candidates.items() for x,y,cl in items if not cl]
            keep_set = set(random.Random(seed).sample(all_emp, target))
        n_tiles = n_labeled = 0
        for img_path, items in candidates.items():
            try:
                with Image.open(img_path) as im:
                    im.load()
                    for x0, y0, clipped in items:
                        if not clipped:
                            if target >= n_emp: pass
                            elif target == 0: continue
                            elif (img_path, x0, y0) not in keep_set: continue
                        stem = f'{img_path.stem}_t{y0}_{x0}'
                        try:
                            tile_img = im.crop((x0, y0, x0+tile_size, y0+tile_size)).convert('RGB')
                            tile_img.save(dst_img/f'{stem}.jpg', 'JPEG', quality=jpeg_quality)
                        except Exception as e: print(f'Tile write error {stem}: {e}'); continue
                        lp = dst_lbl/f'{stem}.txt'
                        if clipped:
                            lp.write_text('\n'.join(_to_yolo_line(*c, w=tile_size, h=tile_size) for c in clipped)+'\n')
                            n_labeled += 1
                        else: lp.write_text('')
                        n_tiles += 1
            except Exception as e: print(f'Error tiling {img_path.name}: {e}')
        stats[split] = {'source_images': n_src, 'tiles': n_tiles, 'labeled_tiles': n_labeled}
        print(f'  [{split}] {n_src} source → {n_tiles} tiles ({n_labeled} labeled)')
    return stats

def ensure_tiled(source_root, extract_dir, tile_size, overlap, min_area,
                 keep_empty, source_config_id, classes):
    """Slice dataset into tiles; cache and reuse if config unchanged."""
    from pathlib import Path
    tiled_root = Path(extract_dir) / 'yolo-tiled'
    marker = tiled_root / '.sliced_for'
    emp_repr = ','.join(f'{k}={v}' for k,v in sorted(keep_empty.items())) if isinstance(keep_empty, dict) else str(keep_empty)
    slice_id = f'source={source_config_id}|tile={tile_size}|overlap={overlap}|min_area={min_area}|empty={emp_repr}'
    if marker.exists() and marker.read_text() == slice_id:
        print(f'[tile] reusing cached tiled dataset at {tiled_root}')
        _write_data_yaml(tiled_root, classes); return str(tiled_root)
    if tiled_root.exists():
        print('[tile] config changed; rebuilding'); shutil.rmtree(tiled_root)
    print(f'[tile] slicing → {tiled_root} (tile={tile_size}, overlap={overlap})')
    slice_dataset(str(source_root), str(tiled_root), tile_size=tile_size,
                  overlap=overlap, min_area=min_area, keep_empty_tiles=keep_empty)
    _write_data_yaml(tiled_root, classes)
    tiled_root.mkdir(parents=True, exist_ok=True)
    marker.write_text(slice_id)
    return str(tiled_root)

# ── Run data prep ─────────────────────────────────────────
assert YOLO_ZIP.exists(), f'yolo.zip not found: {YOLO_ZIP}'
config_id = f"keep={'|'.join(KEEP_CLASSES)}|merge_test={MERGE_TEST_INTO_TRAIN}"
marker = EXTRACT_TO / 'yolo' / '.patched_for'
if marker.exists() and marker.read_text() != config_id:
    print('Config changed since last run; re-extracting...')
    shutil.rmtree(EXTRACT_TO / 'yolo')

dataset_root = extract_zip(YOLO_ZIP, EXTRACT_TO, 'yolo')
if MERGE_TEST_INTO_TRAIN:
    merge_test_into_train(dataset_root)
patch_and_write_yaml(dataset_root, CVAT_CLASSES, KEEP_CLASSES, config_id)

if USE_TILES:
    dataset_root = Path(ensure_tiled(
        source_root=dataset_root,
        extract_dir=EXTRACT_TO,
        tile_size=TILE_SIZE,
        overlap=TILE_OVERLAP,
        min_area=TILE_MIN_AREA,
        keep_empty=KEEP_EMPTY_TILES,
        source_config_id=config_id,
        classes=KEEP_CLASSES,
    ))
    _write_data_yaml(dataset_root, KEEP_CLASSES)

YAML_PATH = dataset_root / 'data.yaml'
print(f'YAML: {YAML_PATH}')


##### Cell 5 — Two-stage training
Stage 1: frozen backbone warm-up. Stage 2: full fine-tune. Best weights copied to `models/yolo_best.pt`.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 5 — TWO-STAGE YOLO TRAINING
# ════════════════════════════════════════════════════════════
from ultralytics import YOLO

# Stage 1: frozen backbone
# optimizer='AdamW' is explicit because Ultralytics' default optimizer='auto'
# silently overrides lr0 and picks its own learning rate (~0.00125),
# ignoring LR_S1 / LR_S2.  AdamW respects the lr0 we set.
print('=== Stage 1: frozen backbone ===')
model = YOLO(MODEL_SIZE)
r1 = model.train(
    data=str(YAML_PATH), epochs=EPOCHS_S1, imgsz=IMG_SIZE, batch=BATCH,
    lr0=LR_S1, optimizer='AdamW', freeze=10, patience=15,
    mosaic=1.0, copy_paste=0.3, mixup=0.1,
    cache=True, seed=SEED, project=str(OUTPUT_DIR), name='stage1', exist_ok=True)
s1_best = OUTPUT_DIR / 'stage1' / 'weights' / 'best.pt'
print(f'Stage 1 best: {s1_best}')
print(f'Stage 1 mAP50: {r1.results_dict.get("metrics/mAP50(B)", "n/a")}')

# Stage 2: full fine-tune
print('\n=== Stage 2: full fine-tune ===')
model2 = YOLO(str(s1_best))
r2 = model2.train(
    data=str(YAML_PATH), epochs=EPOCHS_S2, imgsz=IMG_SIZE, batch=BATCH,
    lr0=LR_S2, optimizer='AdamW', freeze=0, patience=20,
    mosaic=1.0, copy_paste=0.3, mixup=0.1,
    cache='disk', seed=SEED, project=str(OUTPUT_DIR), name='stage2', exist_ok=True)
s2_best = OUTPUT_DIR / 'stage2' / 'weights' / 'best.pt'
print(f'Stage 2 mAP50: {r2.results_dict.get("metrics/mAP50(B)", "n/a")}')

# Copy final weights to models/
dest = MODEL_DIR / 'yolo_best.pt'
shutil.copy(str(s2_best), str(dest))
print(f'\nFinal weights copied to {dest}')
print(f'All run outputs in: {OUTPUT_DIR}')


##### Cell 6 — Evaluation
Runs the trained model on the validation set and prints per-class AP.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 6 — EVALUATION
# ════════════════════════════════════════════════════════════
eval_model = YOLO(str(s2_best))
lines = []
for split in ('val', 'test'):
    m = eval_model.val(data=str(YAML_PATH), split=split, imgsz=IMG_SIZE)
    print(f'\n--- {split.upper()} ---')
    lines.append(f'--- {split.upper()} ---')
    for i, cls in enumerate(KEEP_CLASSES):
        try:
            p=m.box.p[i]; r=m.box.r[i]; f1=2*p*r/max(1e-8,p+r)
            line = f'  {cls:12} P={p:.3f}  R={r:.3f}  F1={f1:.3f}  AP50={m.box.ap50[i]:.3f}'
        except (IndexError, AttributeError):
            line = f'  {cls:12} no detections'
        print(line); lines.append(line)
    summary = f'  mAP50={m.box.map50:.3f}  mAP50-95={m.box.map:.3f}'
    print(summary); lines.append(summary)

report_path = OUTPUT_DIR / 'eval_report.txt'
report_path.write_text('\n'.join(lines))
print(f'\nEval report saved to {report_path}')
print(f'Best model    : {MODEL_DIR}/yolo_best.pt')
